In [ ]:
import psutil
import sys
import importlib.metadata

memory = psutil.virtual_memory()

print("Python:", sys.version.split()[0])
print("Total RAM:", round(memory.total / 2**30, 2), "GiB")
print("Available RAM:", round(memory.available / 2**30, 2), "GiB")

try:
    print("LightGBM:", importlib.metadata.version("lightgbm"))
except importlib.metadata.PackageNotFoundError:
    print("LightGBM: Not installed")

Python: 3.13.16
Total RAM: 12.67 GiB
Available RAM: 11.79 GiB
LightGBM: 4.6.0


In [ ]:
%pip install -q "lightgbm==4.7.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 30.5 MB/s eta 0:00:00


In [ ]:
try:
    print("LightGBM:", importlib.metadata.version("lightgbm"))
except importlib.metadata.PackageNotFoundError:
    print("LightGBM: Not installed")

LightGBM: 4.7.0


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving asi_robustness_cloud_selection.tar.gz to asi_robustness_cloud_selection.tar.gz


In [ ]:
from pathlib import Path

p = Path("/content/asi_robustness_cloud_selection.tar.gz")

print("Uploaded:", p.exists())
if p.exists():
    print("Size:", round(p.stat().st_size / 2**20, 2), "MiB")

Uploaded: True
Size: 50.48 MiB


In [ ]:
import tarfile
from pathlib import Path

archive = Path("/content/asi_robustness_cloud_selection.tar.gz")

with tarfile.open(archive, "r:gz") as tar:
    members = tar.getmembers()

    print("Total entries:", len(members))
    print("\nArchive contents:")

    for member in members[:80]:
        print(
            "DIR " if member.isdir() else "FILE",
            member.name,
            f"({member.size:,} bytes)"
        )

    print("\nTotal uncompressed size:",
          round(sum(m.size for m in members if m.isfile()) / 2**20, 2),
          "MiB")

    unsafe = [
        m.name for m in members
        if m.name.startswith("/")
        or ".." in Path(m.name).parts
        or m.issym()
        or m.islnk()
        or m.isdev()
        or m.isfifo()
    ]

    print("Unsafe entries:", unsafe)

Total entries: 25

Archive contents:
FILE cloud_package/CLOUD_PLAN.md (6,956 bytes)
FILE cloud_package/Dockerfile (335 bytes)
FILE cloud_package/environment.json (367 bytes)
FILE cloud_package/pipeline/asi_pipeline/__init__.py (54 bytes)
FILE cloud_package/pipeline/asi_pipeline/checkpoints.py (3,329 bytes)
FILE cloud_package/pipeline/asi_pipeline/dataset.py (5,769 bytes)
FILE cloud_package/pipeline/asi_pipeline/execution.py (15,523 bytes)
FILE cloud_package/pipeline/asi_pipeline/feasibility.py (3,806 bytes)
FILE cloud_package/pipeline/asi_pipeline/metrics.py (5,559 bytes)
FILE cloud_package/pipeline/asi_pipeline/modeling.py (2,805 bytes)
FILE cloud_package/pipeline/asi_pipeline/primary_data.py (15,898 bytes)
FILE cloud_package/pipeline/asi_pipeline/remote_temporal.py (15,470 bytes)
FILE cloud_package/pipeline/asi_pipeline/safety.py (3,894 bytes)
FILE cloud_package/pipeline/asi_pipeline/scientific_run.py (4,499 bytes)
FILE cloud_package/pipeline/asi_pipeline/temporal.py (2,895 bytes)
FI

In [ ]:
import tarfile
import subprocess
from pathlib import Path

archive = Path("/content/asi_robustness_cloud_selection.tar.gz")

with tarfile.open(archive, "r:gz") as tar:
    tar.extractall("/content", filter="data")

result = subprocess.run(
    ["sha256sum", "-c", "SHA256SUMS"],
    cwd="/content/cloud_package",
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)
print("Checksum verification passed:", result.returncode == 0)

CLOUD_PLAN.md: OK
Dockerfile: OK
environment.json: OK
pipeline/asi_pipeline/__init__.py: OK
pipeline/asi_pipeline/checkpoints.py: OK
pipeline/asi_pipeline/dataset.py: OK
pipeline/asi_pipeline/execution.py: OK
pipeline/asi_pipeline/feasibility.py: OK
pipeline/asi_pipeline/metrics.py: OK
pipeline/asi_pipeline/modeling.py: OK
pipeline/asi_pipeline/primary_data.py: OK
pipeline/asi_pipeline/remote_temporal.py: OK
pipeline/asi_pipeline/safety.py: OK
pipeline/asi_pipeline/scientific_run.py: OK
pipeline/asi_pipeline/temporal.py: OK
pipeline/asi_pipeline/train_cli.py: OK
pipeline/asi_pipeline/windows_monitor.py: OK
protocol_20261020.json: OK
recovery_manifest.json: OK
requirements.txt: OK
run_experiment.py: OK
selection_inputs/train.parquet: OK
selection_inputs/validation.parquet: OK
split_audit.json: OK


Checksum verification passed: True


In [ ]:
from pathlib import Path

root = Path("/content/cloud_package")

print("=== CLOUD PLAN ===")
print((root / "CLOUD_PLAN.md").read_text())

print("\n=== REQUIREMENTS ===")
print((root / "requirements.txt").read_text())

=== CLOUD PLAN ===
# Reproducible cloud continuation — NOT EXECUTED

This is the same post-hoc internal split-sensitivity experiment, not independent external replication. No cloud machine has been provisioned, charged, or trained by this audit.

## Resource and scientific contract

Use a dedicated Linux CPU machine with at least 16 GiB RAM, 4 vCPUs and 20 GiB free disk. No GPU is needed. Run one candidate at a time, with the original two LightGBM threads. Use the container limit below to leave host headroom. If that host fails its memory checks or is OOM-killed, preserve logs and stop; do not reduce rows, features, candidates or boosting budget.

Preserve the existing split files and their SHA-256 hashes; do not reconstruct the split. Use model seed 20260927, split seed 20261020, original M0 R+P+A and M1 R+P+A+O, train-only categorical encoding, all 12 candidates per role, 200 maximum rounds, 30-round early stopping, and lowest-candidate-index tie breaking. No train+validation refit i

In [ ]:
import psutil
import subprocess
from pathlib import Path

root = Path("/content/cloud_package")

mem = psutil.virtual_memory()
print("Total RAM:", round(mem.total / 2**30, 2), "GiB")
print("Available RAM:", round(mem.available / 2**30, 2), "GiB")

result = subprocess.run(
    ["python", "run_experiment.py", "--help"],
    cwd=root,
    capture_output=True,
    text=True
)

print("\n=== RUNNER HELP ===")
print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

print("\n=== DOCKERFILE ===")
print((root / "Dockerfile").read_text())

Total RAM: 12.67 GiB
Available RAM: 11.48 GiB

=== RUNNER HELP ===
usage: run_experiment.py [-h] --out OUT [--test TEST]
                         {select,verify-freeze,evaluate}

positional arguments:
  {select,verify-freeze,evaluate}

options:
  -h, --help            show this help message and exit
  --out OUT
  --test TEST


Exit code: 0

=== DOCKERFILE ===
FROM python:3.12.2-slim-bookworm
RUN apt-get update && apt-get install -y --no-install-recommends libgomp1 && rm -rf /var/lib/apt/lists/*
COPY requirements.txt /tmp/requirements.txt
RUN pip install --no-cache-dir -r /tmp/requirements.txt
ENV PYTHONDONTWRITEBYTECODE=1
WORKDIR /app
ENTRYPOINT ["python", "-u", "/app/run_experiment.py"]



In [ ]:
import sys
import importlib.metadata as md

print("Python:", sys.version)

for package in [
    "lightgbm", "numpy", "pandas", "pyarrow",
    "duckdb", "scipy", "PyYAML", "psutil"
]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
lightgbm: 4.7.0
numpy: 2.1.3
pandas: 2.2.3
pyarrow: 23.0.1
duckdb: 1.3.2
scipy: 1.16.3
PyYAML: 6.0.3
psutil: 5.9.5


In [ ]:
%pip install -q -r /content/cloud_package/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 55.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.5/287.5 kB 21.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.10.15 requires numpy>=2.1, but you have numpy 2.0.1 which is incompatible

In [ ]:
import sys
import importlib.metadata as md

print("Python:", sys.version.split()[0])

expected = {
    "lightgbm": "4.7.0",
    "numpy": "2.0.1",
    "pandas": "2.2.3",
    "pyarrow": "25.0.1",
    "duckdb": "1.5.5",
    "scipy": "1.15.1",
    "PyYAML": "6.0.2",
    "psutil": "6.1.1",
}

for name, version in expected.items():
    try:
        installed = md.version(name)
        print(f"{name}: {installed} {'OK' if installed == version else 'MISMATCH'}")
    except md.PackageNotFoundError:
        print(f"{name}: NOT INSTALLED")

Python: 3.13.16
lightgbm: 4.7.0 OK
numpy: 2.0.1 OK
pandas: 2.2.3 OK
pyarrow: 25.0.1 OK
duckdb: 1.5.5 OK
scipy: 1.15.1 OK
PyYAML: 6.0.2 OK
psutil: 6.1.1 OK


In [ ]:
import numpy
import pandas
import pyarrow
import duckdb
import scipy
import lightgbm
import psutil
import yaml

print("All experiment libraries imported successfully.")

All experiment libraries imported successfully.


In [ ]:
import subprocess
import threading
import time
import psutil
import sys
from pathlib import Path

root = Path("/content/cloud_package")
base = Path("/content/cloud-results")
base.mkdir(parents=True, exist_ok=True)

n = 1
while (base / f"run_colab_{n:02d}").exists():
    n += 1

out = base / f"run_colab_{n:02d}"
logs = Path("/content/cloud-logs")
logs.mkdir(parents=True, exist_ok=True)

print("Fresh output directory:", out)

def monitor_memory(stop_event):
    with (logs / "memory.csv").open("w") as f:
        f.write("timestamp,available_gib,used_percent\n")
        while not stop_event.is_set():
            mem = psutil.virtual_memory()
            f.write(
                f"{time.time():.0f},"
                f"{mem.available / 2**30:.3f},"
                f"{mem.percent:.1f}\n"
            )
            f.flush()
            stop_event.wait(5)

stop_event = threading.Event()
monitor = threading.Thread(
    target=monitor_memory,
    args=(stop_event,),
    daemon=True
)

monitor.start()

print("Starting validation-only selection...", flush=True)
print("Results:", out, flush=True)

try:
    with (logs / "selection.log").open("w") as logfile:
        process = subprocess.Popen(
            [
                sys.executable, "-u",
                str(root / "run_experiment.py"),
                "select",
                "--out", str(out),
            ],
            cwd=root,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            logfile.write(line)
            logfile.flush()

        exit_code = process.wait()

finally:
    stop_event.set()
    monitor.join(timeout=10)

print("\nSelection exit code:", exit_code)
print("Memory log:", logs / "memory.csv")
print("Training log:", logs / "selection.log")

Fresh output directory: /content/cloud-results/run_colab_02
Starting validation-only selection...
Results: /content/cloud-results/run_colab_02
{"role": "m0", "candidate_id": 1, "validation_log_mae": 0.8141400482411273, "best_iteration": 21, "parameters": {"objective": "regression_l1", "metric": "l1", "learning_rate": 0.05, "bagging_fraction": 0.8, "bagging_freq": 1, "verbosity": -1, "num_threads": 2, "deterministic": true, "force_col_wise": true, "feature_pre_filter": false, "num_leaves": 15, "min_data_in_leaf": 20, "feature_fraction": 0.8, "seed": 20260927, "bagging_seed": 20260927, "feature_fraction_seed": 20260927, "data_random_seed": 20260927}, "training_seconds": 18.500564120000035, "prediction_seconds": 0.5197178009998424}
{"role": "m0", "candidate_id": 2, "validation_log_mae": 0.7852658401015193, "best_iteration": 141, "parameters": {"objective": "regression_l1", "metric": "l1", "learning_rate": 0.05, "bagging_fraction": 0.8, "bagging_freq": 1, "verbosity": -1, "num_threads": 2,

In [ ]:
from pathlib import Path

out = Path("/content/cloud-results/run")

print("Exists:", out.exists())
print("Contents:", list(out.iterdir()) if out.is_dir() else "None")

Exists: True
Contents: []


In [ ]:
from pathlib import Path

base = Path("/content/cloud-results")

for name in ["run", "run_colab_01", "run_colab_02"]:
    p = base / name
    print(name, "| exists:", p.exists(),
          "| contents:", [x.name for x in p.iterdir()] if p.is_dir() else "N/A")

run | exists: True | contents: []
run_colab_01 | exists: True | contents: []
run_colab_02 | exists: False | contents: N/A


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "/content/cloud_package/run_experiment.py",
        "verify-freeze",
        "--out",
        "/content/cloud-results/run_colab_02",
    ],
    cwd="/content/cloud_package",
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
print("Freeze verification passed:", result.returncode == 0)

Both model selections and all frozen artifact hashes verified.


Freeze verification passed: True


In [ ]:
from pathlib import Path
import shutil

archive = shutil.make_archive(
    "/content/asi_robustness_colab_selection",
    "zip",
    root_dir="/content",
    base_dir="cloud-results",
)

print("Saved:", archive)
print("Size:", round(Path(archive).stat().st_size / 2**20, 2), "MiB")

Saved: /content/asi_robustness_colab_selection.zip
Size: 0.37 MiB


In [ ]:
from google.colab import files
files.download("/content/asi_robustness_colab_selection.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/asi_robustness_colab_logs",
    "zip",
    root_dir="/content",
    base_dir="cloud-logs",
)

files.download("/content/asi_robustness_colab_logs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

root = Path("/content/cloud_package")
output = Path("/content/asi_colab_source_audit.zip")

if not root.is_dir():
    raise FileNotFoundError(
        "cloud_package is missing. Restore the original tar.gz first."
    )

with ZipFile(output, "w", ZIP_DEFLATED) as z:
    for path in root.rglob("*"):
        if path.is_file() and path.suffix in {".py", ".yaml", ".yml", ".json", ".txt", ".md"}:
            if path.stat().st_size <= 2_000_000:
                z.write(path, path.relative_to(root.parent))

print("Created:", output)
print("Size:", round(output.stat().st_size / 1024, 1), "KB")
files.download(str(output))

FileNotFoundError: cloud_package is missing. Restore the original tar.gz first.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
import tarfile

print("Upload asi_robustness_cloud_selection.tar.gz")
uploaded = files.upload()

archive_name = next(
    (name for name in uploaded if name.endswith((".tar.gz", ".tgz"))),
    None
)

if archive_name is None:
    raise ValueError("Please upload the original .tar.gz package.")

output = Path("/content/asi_colab_source_audit.zip")
allowed = (".py", ".yaml", ".yml", ".json", ".txt", ".md")

count = 0

with tarfile.open(archive_name, "r:gz") as tar:
    with ZipFile(output, "w", ZIP_DEFLATED) as zip_file:
        for member in tar:
            if not member.isfile():
                continue

            path = Path(member.name)

            # Avoid unsafe paths and irrelevant files
            if path.is_absolute() or ".." in path.parts:
                continue
            if path.suffix.lower() not in allowed:
                continue
            if member.size > 2_000_000:
                continue

            extracted = tar.extractfile(member)
            if extracted is None:
                continue

            zip_file.writestr(
                str(path),
                extracted.read()
            )
            count += 1

print("Source files collected:", count)
print("ZIP:", output)
print("Size:", round(output.stat().st_size / 1024, 1), "KB")

files.download(str(output))

Upload asi_robustness_cloud_selection.tar.gz


Saving asi_robustness_cloud_selection.tar.gz to asi_robustness_cloud_selection.tar.gz
Source files collected: 21
ZIP: /content/asi_colab_source_audit.zip
Size: 47.0 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import tarfile
from pathlib import Path

archives = list(Path("/content").rglob("asi_robustness_cloud_selection.tar.gz"))

if not archives:
    print("Archive not found under /content")
else:
    archive = archives[0]
    print("Archive found:", archive)

    with tarfile.open(archive, "r:gz") as tar:
        members = [m for m in tar if m.isfile()]

        print("\nDATA FILES:")
        for m in members:
            if m.name.lower().endswith((".parquet", ".csv", ".arrow")):
                print(m.name, f"({m.size / 1024**2:.2f} MB)")

        print("\nSPLIT AND CONFIGURATION FILES:")
        for m in members:
            if any(s in m.name.lower() for s in
                   ["split", "manifest", "config", "dataset"]):
                print(m.name)

Archive found: /content/asi_robustness_cloud_selection.tar.gz

DATA FILES:
cloud_package/selection_inputs/train.parquet (41.37 MB)
cloud_package/selection_inputs/validation.parquet (9.38 MB)

SPLIT AND CONFIGURATION FILES:
cloud_package/pipeline/asi_pipeline/dataset.py
cloud_package/recovery_manifest.json
cloud_package/split_audit.json


In [ ]:
import tarfile
from pathlib import Path

archive = Path("/content/asi_robustness_cloud_selection.tar.gz")
destination = Path("/content/asi_diagnostic_inputs")
destination.mkdir(exist_ok=True)

wanted = {
    "cloud_package/selection_inputs/train.parquet",
    "cloud_package/selection_inputs/validation.parquet",
}

with tarfile.open(archive, "r:gz") as tar:
    for member in tar:
        if member.name not in wanted:
            continue

        target = destination / Path(member.name).name

        with tar.extractfile(member) as source, open(target, "wb") as output:
            while chunk := source.read(1024 * 1024):
                output.write(chunk)

        print("Extracted:", target)

Extracted: /content/asi_diagnostic_inputs/train.parquet
Extracted: /content/asi_diagnostic_inputs/validation.parquet


In [ ]:
import pandas as pd
import numpy as np

train = pd.read_parquet("/content/asi_diagnostic_inputs/train.parquet")
valid = pd.read_parquet("/content/asi_diagnostic_inputs/validation.parquet")

print("TRAIN:", train.shape)
print("VALIDATION:", valid.shape)

for name, df in [("TRAIN", train), ("VALIDATION", valid)]:
    print(f"\n===== {name} =====")

    print("\nWorkload-type distribution (%):")
    print(
        df["job_type_public"]
        .value_counts(dropna=False, normalize=True)
        .mul(100)
        .round(2)
    )

    print("\nDuration quantiles (hours):")
    print(
        df["duration_hours"]
        .quantile([0.5, 0.9, 0.95, 0.99, 0.999])
    )

    print("\nDuration by workload type:")
    print(
        df.groupby("job_type_public")["duration_hours"]
        .agg(["count", "median", "mean"])
        .round(3)
    )

    print("\nUnique workloads:", df["workload_id"].nunique())

print("\nTrain/validation workload overlap:")
print(
    len(
        set(train["workload_id"].dropna())
        & set(valid["workload_id"].dropna())
    )
)

TRAIN: (3223597, 11)
VALIDATION: (752865, 11)

===== TRAIN =====

Workload-type distribution (%):
job_type_public
training             63.70
offline_inference    23.16
online_inference     12.49
dev                   0.65
Name: proportion, dtype: float64

Duration quantiles (hours):
0.500       1.000000
0.900      13.994444
0.950      31.755556
0.990     455.994444
0.999    1344.000000
Name: duration_hours, dtype: float64

Duration by workload type:
                     count  median     mean
job_type_public                            
dev                  21006   9.550   75.320
offline_inference   746545   0.356    1.874
online_inference    402542   2.722  102.186
training           2053504   1.000    4.403

Unique workloads: 330836

===== VALIDATION =====

Workload-type distribution (%):
job_type_public
training             59.50
online_inference     20.50
offline_inference    19.48
dev                   0.52
Name: proportion, dtype: float64

Duration quantiles (hours):
0.500       0

In [ ]:
for name, df in [("TRAIN", train), ("VALIDATION", valid)]:
    online = df[df["job_type_public"] == "online_inference"]

    print(f"\n===== {name}: ONLINE INFERENCE =====")

    print("Duration quantiles (hours):")
    print(
        online["duration_hours"]
        .quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99])
        .round(4)
    )

    counts = online.groupby("workload_id").size()

    print("\nUnique workloads:", len(counts))
    print("Executions per workload:")
    print(counts.quantile([0.5, 0.9, 0.99]).round(2))

    print(
        "Share of executions from top 1% largest workloads:",
        round(
            100 * counts.nlargest(max(1, int(len(counts) * 0.01))).sum()
            / len(online),
            2
        ),
        "%"
    )


===== TRAIN: ONLINE INFERENCE =====
Duration quantiles (hours):
0.10       0.0222
0.25       0.0944
0.50       2.7222
0.75      63.7833
0.90     342.4161
0.99    1135.5222
Name: duration_hours, dtype: float64

Unique workloads: 28877
Executions per workload:
0.50      1.00
0.90      6.00
0.99    173.24
dtype: float64
Share of executions from top 1% largest workloads: 71.47 %

===== VALIDATION: ONLINE INFERENCE =====
Duration quantiles (hours):
0.10       0.0278
0.25       0.0333
0.50       0.0389
0.75      11.1556
0.90     120.0000
0.99    1031.9833
Name: duration_hours, dtype: float64

Unique workloads: 6174
Executions per workload:
0.50      1.00
0.90      6.00
0.99    168.24
dtype: float64
Share of executions from top 1% largest workloads: 83.78 %


In [ ]:
print("TRAIN COLUMNS:")
print(train.columns.tolist())

print("\nVALIDATION COLUMNS:")
print(valid.columns.tolist())

TRAIN COLUMNS:
['workload_id', 'duration_hours', 'gpu_request', 'gpu_spec_public', 'priority_class', 'model_type_public', 'is_genai_request', 'job_type_public', 'row_ordinal', 'split', 'grouped_split']

VALIDATION COLUMNS:
['workload_id', 'duration_hours', 'gpu_request', 'gpu_spec_public', 'priority_class', 'model_type_public', 'is_genai_request', 'job_type_public', 'row_ordinal', 'split', 'grouped_split']


In [ ]:
import tarfile
import json

archive = "/content/asi_robustness_cloud_selection.tar.gz"

with tarfile.open(archive, "r:gz") as tar:
    for name in [
        "cloud_package/recovery_manifest.json",
        "cloud_package/split_audit.json"
    ]:
        print("\n====", name, "====")
        f = tar.extractfile(name)
        data = json.load(f)
        print(json.dumps(data, indent=2)[:12000])


==== cloud_package/recovery_manifest.json ====
[
  {
    "path": "checkpoint/archive_tools/verify_checkpoint.py",
    "bytes": 1000,
    "sha256": "d74d679c0b28cf0362fcf4ca9b78bdb74323656cc933a2da472f6d86cbe26252",
    "original_source_path": "C:\\Users\\adars\\OneDrive\\Desktop\\ASI_Research\\resume_31210\\working_checkpoint\\archive_tools\\verify_checkpoint.py"
  },
  {
    "path": "checkpoint/asi_pipeline/__init__.py",
    "bytes": 54,
    "sha256": "05f59b72c95fe11e48825a04f143c8db97dbab934a3c8f90743fb610c23ec6a4",
    "original_source_path": "C:\\Users\\adars\\OneDrive\\Desktop\\ASI_Research\\resume_31210\\working_checkpoint\\asi_pipeline\\__init__.py"
  },
  {
    "path": "checkpoint/asi_pipeline/checkpoints.py",
    "bytes": 3329,
    "sha256": "0d95523289640ef4266e3ab8fc4d1fe654ac5605a67d2888c30b4ac0958307e2",
    "original_source_path": "C:\\Users\\adars\\OneDrive\\Desktop\\ASI_Research\\resume_31210\\working_checkpoint\\asi_pipeline\\checkpoints.py"
  },
  {
    "path": "che

In [ ]:
import tarfile
import json

archive = "/content/asi_robustness_cloud_selection.tar.gz"
name = "cloud_package/checkpoint/windows_run/canonical/manifest.json"

with tarfile.open(archive, "r:gz") as tar:
    member = tar.getmember(name)
    with tar.extractfile(member) as f:
        manifest = json.load(f)

print(json.dumps(manifest, indent=2)[:16000])

KeyError: "filename 'cloud_package/checkpoint/windows_run/canonical/manifest.json' not found"

In [ ]:
from pathlib import Path
import json

root = Path("/content")

for p in sorted(root.rglob("selection.json")):
    if "run_colab_02" not in str(p):
        continue

    print("\nFILE:", p)
    data = json.loads(p.read_text())
    print(json.dumps(data, indent=2)[:5000])

print("\nSAVED MODEL FILES:")
for p in sorted(root.rglob("model.txt")):
    if "run_colab_02" in str(p):
        print(p)


SAVED MODEL FILES:


In [ ]:
from pathlib import Path
import zipfile
import tarfile

for p in Path("/content").rglob("*"):
    if not p.is_file():
        continue

    if p.suffix == ".zip":
        print("\nZIP:", p)
        with zipfile.ZipFile(p) as z:
            for name in z.namelist():
                if name.endswith(("model.txt", "selection.json")):
                    print(" ", name)

    elif p.name.endswith((".tar.gz", ".tgz")):
        print("\nTAR:", p)
        with tarfile.open(p, "r:gz") as t:
            for member in t:
                if member.name.endswith(("model.txt", "selection.json")):
                    print(" ", member.name)


TAR: /content/asi_robustness_cloud_selection.tar.gz

ZIP: /content/asi_colab_source_audit.zip


In [ ]:
import zipfile
from pathlib import Path

p = Path("/content/asi_robustness_colab_selection.zip")

with zipfile.ZipFile(p) as z:
    for name in z.namelist():
        if name.endswith(("model.txt", "selection.json")):
            print(name)

cloud-results/run_colab_02/m1/model.txt
cloud-results/run_colab_02/m1/selection.json
cloud-results/run_colab_02/m1/candidates/11/model.txt
cloud-results/run_colab_02/m1/candidates/05/model.txt
cloud-results/run_colab_02/m1/candidates/08/model.txt
cloud-results/run_colab_02/m1/candidates/06/model.txt
cloud-results/run_colab_02/m1/candidates/02/model.txt
cloud-results/run_colab_02/m1/candidates/01/model.txt
cloud-results/run_colab_02/m1/candidates/09/model.txt
cloud-results/run_colab_02/m1/candidates/03/model.txt
cloud-results/run_colab_02/m1/candidates/12/model.txt
cloud-results/run_colab_02/m1/candidates/10/model.txt
cloud-results/run_colab_02/m1/candidates/07/model.txt
cloud-results/run_colab_02/m1/candidates/04/model.txt
cloud-results/run_colab_02/m0/model.txt
cloud-results/run_colab_02/m0/selection.json
cloud-results/run_colab_02/m0/candidates/11/model.txt
cloud-results/run_colab_02/m0/candidates/05/model.txt
cloud-results/run_colab_02/m0/candidates/08/model.txt
cloud-results/run_co

In [ ]:
import zipfile
import json
import re

archive = "/content/asi_robustness_colab_selection.zip"

with zipfile.ZipFile(archive) as z:
    for model in ["m0", "m1"]:
        prefix = f"cloud-results/run_colab_02/{model}/"

        selection = json.loads(
            z.read(prefix + "selection.json")
        )
        model_text = z.read(prefix + "model.txt").decode("utf-8")

        trees = re.findall(
            r"^Tree=\d+$",
            model_text,
            flags=re.MULTILINE
        )

        print(f"\n===== {model.upper()} =====")
        print("Saved tree count:", len(trees))
        print("Selection metadata:")
        print(json.dumps(selection, indent=2)[:5000])


===== M0 =====
Saved tree count: 141
Selection metadata:
{
  "features": [
    "gpu_request",
    "gpu_spec_public",
    "priority_class",
    "model_type_public",
    "is_genai_request"
  ],
  "records": [
    {
      "candidate_id": 1,
      "validation_log_mae": 0.8141400482411273,
      "best_iteration": 21,
      "parameters": {
        "objective": "regression_l1",
        "metric": "l1",
        "learning_rate": 0.05,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "verbosity": -1,
        "num_threads": 2,
        "deterministic": true,
        "force_col_wise": true,
        "feature_pre_filter": false,
        "num_leaves": 15,
        "min_data_in_leaf": 20,
        "feature_fraction": 0.8,
        "seed": 20260927,
        "bagging_seed": 20260927,
        "feature_fraction_seed": 20260927,
        "data_random_seed": 20260927
      }
    },
    {
      "candidate_id": 2,
      "validation_log_mae": 0.7852658401015193,
      "best_iteration": 141,
     

In [ ]:
import zipfile
import lightgbm as lgb
import tempfile
from pathlib import Path

archive = "/content/asi_robustness_colab_selection.zip"
member = "cloud-results/run_colab_02/m1/model.txt"

with zipfile.ZipFile(archive) as z:
    model_bytes = z.read(member)

with tempfile.TemporaryDirectory() as d:
    path = Path(d) / "m1_model.txt"
    path.write_bytes(model_bytes)
    model = lgb.Booster(model_file=str(path))

    print("Number of trees:", model.num_trees())
    print("\nFeature importance (split counts):")

    for feature, count in zip(
        model.feature_name(),
        model.feature_importance(importance_type="split")
    ):
        print(f"{feature}: {count}")

    print("\nFeature importance (gain):")

    for feature, gain in zip(
        model.feature_name(),
        model.feature_importance(importance_type="gain")
    ):
        print(f"{feature}: {gain:.2f}")

Number of trees: 5

Feature importance (split counts):
gpu_request: 13
gpu_spec_public: 31
priority_class: 13
model_type_public: 6
is_genai_request: 2
job_type_public: 5

Feature importance (gain):
gpu_request: 309162.87
gpu_spec_public: 216339.37
priority_class: 119099.41
model_type_public: 273906.26
is_genai_request: 34203.32
job_type_public: 259986.19


In [ ]:
import zipfile
import tempfile
from pathlib import Path
import lightgbm as lgb
import numpy as np
import pandas as pd
import json

archive = "/content/asi_robustness_colab_selection.zip"
valid_path = "/content/asi_diagnostic_inputs/validation.parquet"

valid = pd.read_parquet(valid_path)

with zipfile.ZipFile(archive) as z:
    for name in ["m0", "m1"]:
        prefix = f"cloud-results/run_colab_02/{name}/"

        encoding = json.loads(z.read(prefix + "encoding.json"))
        model_bytes = z.read(prefix + "model.txt")

        print(f"\n{name.upper()}")
        print("Encoding structure:", list(encoding.keys()))
        print("Model bytes:", len(model_bytes))


M0
Encoding structure: ['gpu_spec_public', 'priority_class', 'model_type_public', 'is_genai_request']
Model bytes: 221409

M1
Encoding structure: ['gpu_spec_public', 'priority_class', 'model_type_public', 'is_genai_request', 'job_type_public']
Model bytes: 10538


In [ ]:
import zipfile
import json

archive = "/content/asi_robustness_colab_selection.zip"

with zipfile.ZipFile(archive) as z:
    encodings = {}

    for name in ["m0", "m1"]:
        path = f"cloud-results/run_colab_02/{name}/encoding.json"
        encodings[name] = json.loads(z.read(path))

    print("SHARED FEATURE ENCODINGS")
    for feature in encodings["m0"]:
        same = encodings["m0"][feature] == encodings["m1"][feature]
        print(f"{feature}: {'IDENTICAL' if same else 'DIFFERENT'}")

    print("\nM1 JOB-TYPE ENCODING")
    print(json.dumps(encodings["m1"]["job_type_public"], indent=2))

    print("\nEXAMPLE SHARED ENCODING")
    print(json.dumps(encodings["m1"]["gpu_spec_public"], indent=2)[:1500])

SHARED FEATURE ENCODINGS
gpu_spec_public: IDENTICAL
priority_class: IDENTICAL
model_type_public: IDENTICAL
is_genai_request: IDENTICAL

M1 JOB-TYPE ENCODING
[
  "dev",
  "offline_inference",
  "online_inference",
  "training"
]

EXAMPLE SHARED ENCODING
[
  "A10",
  "A100",
  "A30",
  "A800",
  "H20",
  "H20-141GB",
  "H800",
  "L20",
  "XPU-A",
  "XPU-B",
  "XPU-C",
  "XPU-D",
  "XPU-E"
]


In [ ]:
import zipfile
import json
import tempfile
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

archive = "/content/asi_robustness_colab_selection.zip"
valid = pd.read_parquet(
    "/content/asi_diagnostic_inputs/validation.parquet"
)

y = np.log1p(valid["duration_hours"].to_numpy(dtype=float))

with zipfile.ZipFile(archive) as z:
    for name in ["m0", "m1"]:
        prefix = f"cloud-results/run_colab_02/{name}/"
        encodings = json.loads(z.read(prefix + "encoding.json"))

        with tempfile.TemporaryDirectory() as tmp:
            path = Path(tmp) / "model.txt"
            path.write_bytes(z.read(prefix + "model.txt"))
            model = lgb.Booster(model_file=str(path))

            features = model.feature_name()
            X = valid[features].copy()

            for feature, categories in encodings.items():
                mapping = {
                    str(category): i
                    for i, category in enumerate(categories)
                }
                X[feature] = (
                    X[feature]
                    .astype("string")
                    .map(mapping)
                    .fillna(-1)
                    .astype("int32")
                )

            X["gpu_request"] = pd.to_numeric(
                X["gpu_request"], errors="coerce"
            )

            print(f"\n===== {name.upper()} =====")
            print("Saved trees:", model.num_trees())
            print("Iteration | Validation LogMAE")

            checkpoints = (
                list(range(1, model.num_trees() + 1))
                if name == "m1"
                else sorted(set(
                    [1, 2, 3, 5, 10, 20, 50, 100,
                     model.num_trees()]
                ))
            )

            for iteration in checkpoints:
                if iteration > model.num_trees():
                    continue

                pred = model.predict(
                    X,
                    num_iteration=iteration
                )
                loss = np.mean(np.abs(y - pred))
                print(f"{iteration:9d} | {loss:.8f}")


===== M0 =====
Saved trees: 141
Iteration | Validation LogMAE
        1 | 0.83838921
        2 | 0.83295831
        3 | 0.83069440
        5 | 0.82411496
       10 | 0.80946715
       20 | 0.79901624
       50 | 0.79388891
      100 | 0.78751783
      141 | 0.78526584

===== M1 =====
Saved trees: 5
Iteration | Validation LogMAE
        1 | 0.83555806
        2 | 0.83316588
        3 | 0.83047750
        4 | 0.82573273
        5 | 0.82379590


In [ ]:
import zipfile
import json

archive = "/content/asi_robustness_colab_selection.zip"
path = "cloud-results/run_colab_02/m1/selection.json"

with zipfile.ZipFile(archive) as z:
    selection = json.loads(z.read(path))

print(json.dumps(selection, indent=2))

{
  "features": [
    "gpu_request",
    "gpu_spec_public",
    "priority_class",
    "model_type_public",
    "is_genai_request",
    "job_type_public"
  ],
  "records": [
    {
      "candidate_id": 1,
      "validation_log_mae": 0.8237959010870514,
      "best_iteration": 5,
      "parameters": {
        "objective": "regression_l1",
        "metric": "l1",
        "learning_rate": 0.05,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "verbosity": -1,
        "num_threads": 2,
        "deterministic": true,
        "force_col_wise": true,
        "feature_pre_filter": false,
        "num_leaves": 15,
        "min_data_in_leaf": 20,
        "feature_fraction": 0.8,
        "seed": 20260927,
        "bagging_seed": 20260927,
        "feature_fraction_seed": 20260927,
        "data_random_seed": 20260927
      }
    },
    {
      "candidate_id": 2,
      "validation_log_mae": 0.8311935150661526,
      "best_iteration": 7,
      "parameters": {
        "objective": 